# Evading AI-Generated Text Detection -- Activation Steering and Lexical Variance

**Kaggle Competition Solution**

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

This implementation establishes a multi-stage evasion pipeline. It uses 
coordinated activation steering at Layer 12 across multiple SAE 
latent features, combined with high-entropy sampling and deep lexical 
intervention to maximize output variance and circumvent statistical 
classifiers (like DeBERTa-v3-large).

**Key Enhancements (Score Maximization):**
- **Ensemble Steering**: Suppressing features related to academic tone, formal structure, and formulaic logic.
- **High-Entropy Generation**: temperature 1.3 + top_k 40 distribution sampling.
- **Deep Post-Processing**: Replacement of AI-specific vocabulary ("AI-isms") with human-like discourse markers.

---


## 1. Setup and Dependencies

Standard stack for transformer inference. We use the Kaggle Secrets 
API for authenticated access to the Gemma gated weights.


In [0]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from typing import List, Union, Dict
from huggingface_hub import hf_hub_download, login
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from pathlib import Path

# suppress tokenization parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from kaggle_secrets import UserSecretsClient
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("WARNING: HF_TOKEN secret not found.")


## 2. Model and SAE Definitions

Custom wrappers for the LLM and Sparse Autoencoder. The 
HookedModelWrapper stabilizes inference by managing padding 
and ensuring inputs match the model's physical device layout.


In [0]:
class HookedModelWrapper(nn.Module):
    def __init__(self, model, tokenizer):
        super().__init__()
        self.model = model
        self.model.eval()
        self.tokenizer = tokenizer
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def to_tokens(self, text: Union[str, List[str]], prepend_bos: bool = True):
        if isinstance(text, str): text = [text]
        tokens = self.tokenizer(text, return_tensors="pt", padding=True, add_special_tokens=prepend_bos)
        return tokens.input_ids.to(self.model.device)

    def generate(self, prompts: List[str], max_new_tokens: int = 128, **kwargs):
        input_ids = self.to_tokens(prompts)
        out = self.model.generate(
            input_ids, 
            max_new_tokens=max_new_tokens,
            pad_token_id=self.tokenizer.pad_token_id,
            **kwargs
        )
        input_len = input_ids.shape[1]
        return self.tokenizer.batch_decode(out[:, input_len:], skip_special_tokens=True)

class ManualSAE(nn.Module):
    def __init__(self, params_path: str, device: str = "cpu"):
        super().__init__()
        data = np.load(params_path)

        w_enc_key = "W_enc" if "W_enc" in data.files else "w_enc"
        w_dec_key = "W_dec" if "W_dec" in data.files else "w_dec"

        self.W_enc = nn.Parameter(torch.tensor(data[w_enc_key], dtype=torch.float32))
        self.b_enc = nn.Parameter(torch.tensor(data["b_enc"], dtype=torch.float32))
        self.W_dec = nn.Parameter(torch.tensor(data[w_dec_key], dtype=torch.float32))
        self.b_dec = nn.Parameter(torch.tensor(data["b_dec"], dtype=torch.float32))
        self.to(device)

    @staticmethod
    def from_pretrained(release: str, sae_id: str, device: str = "cpu") -> "ManualSAE":
        repo_id = f"google/{release}"
        filename = f"{sae_id}/params.npz"
        local_path = hf_hub_download(repo_id=repo_id, filename=filename)
        return ManualSAE(local_path, device=device)


## 3. Configuration and Constants

Hyperparameters optimized for evasion. We use Layer 12 (the most 
validated layer for Gemma-2-2B steering) and ensemble magnitudes 
(-7.5) to break AI-like statistical patterns while preserving logic.


In [0]:
class Config:
    model_name = "google/gemma-2-2b"
    release = "gemma-scope-2b-pt-res"
    layer = 12
    sae_width = "16k"
    sae_l0 = 82

    # high-entropy sampling increases variance significantly
    temperature = 1.3
    top_p = 0.95
    top_k = 40

    # automated path discovery for Kaggle datasets
    try:
        data_path = str(next(Path("/kaggle/input").rglob("test.csv")).parent)
    except:
        data_path = "/kaggle/input/competitions/cyprus-spring-2026-evading-ai-generated-text-detection"

    test_dataset_name = "test.csv"
    # Increased batch size to ensure < 15 min runtime
    batch_size = 16 

# steering configuration: suppression ensemble for Layer 12
STEERING_FEATURES = {
    1205: -7.5, # academic tone
    413: -5.0,  # formulaic pattern
    1421: -5.0, # formal structure
}

def load_assets():
    token = HF_TOKEN
    tokenizer = AutoTokenizer.from_pretrained(Config.model_name, token=token)
    model = AutoModelForCausalLM.from_pretrained(
        Config.model_name, token=token, device_map="auto", torch_dtype=torch.bfloat16
    )

    sae = ManualSAE.from_pretrained(
        release=Config.release,
        sae_id=f"layer_{Config.layer}/width_{Config.sae_width}/average_l0_{Config.sae_l0}",
        device="cpu" 
    )
    return model, tokenizer, sae

hf_model, tokenizer, sae = load_assets()
model = HookedModelWrapper(hf_model, tokenizer)


## 4. Feature Intervention Logic

Coordinated multi-feature steering across Layer 12 for maximum 
distributional shift away from formulaic internal weights.


In [0]:
# combine multiple feature directions into a single intervention vector
combined_direction = torch.zeros_like(sae.W_dec[0])
for feat_id, magnitude in STEERING_FEATURES.items():
    combined_direction += sae.W_dec[feat_id] * magnitude

# shift to model backend device and cast to BFloat16 to match model internals
combined_direction = combined_direction.to(hf_model.device).to(torch.bfloat16)

def residual_steering_hook(module, input, kwargs, output):
    hidden_states = output[0] if isinstance(output, tuple) else output

    # apply latent steering intervention
    hidden_states = hidden_states + combined_direction.to(hidden_states.device)

    if isinstance(output, tuple):
        return (hidden_states,) + output[1:]
    return hidden_states

target_layer = hf_model.model.layers[Config.layer]
hook_handle = target_layer.register_forward_hook(residual_steering_hook, with_kwargs=True)

print(f"Activation intervention initialized on Layer {Config.layer}")


## 5. Lexical Intervention (Entropy Maximization)

Deep post-processing to replace overused formalisms (AI-isms) 
with neutral or casual human-like discourse markers.


In [0]:
# Lexical substitution map to break formal AI personas
AI_REPLACEMENTS = {
    "it is important to note": "note that",
    "it is worth noting": "notably",
    "in conclusion,": "so,",
    "furthermore,": "also,",
    "moreover,": "plus,",
    "leverage": "use",
    "tapestry": "mix",
    "comprehensive": "thorough",
    "essential to": "you need to",
    "to summarize,": "basically,",
}

# human typography uses contractions more heavily than default LLM outputs
CONTRACTIONS = {
    "do not": "don't", "does not": "doesn't", "did not": "didn't",
    "it is": "it's", "that is": "that's",
}

def apply_lexical_intervention(text: str) -> str:
    # 1. Replace formal AI-isms with neutral alternatives
    for phrase, replacement in AI_REPLACEMENTS.items():
        text = re.sub(re.escape(phrase), replacement, text, flags=re.IGNORECASE)

    # 2. Inject contractions probabilistically (90% frequency)
    for full, contracted in CONTRACTIONS.items():
        def _contract(m): return contracted if random.random() < 0.90 else m.group(0)
        text = re.sub(r'\b' + re.escape(full) + r'\b', _contract, text, flags=re.IGNORECASE)

    # 3. Final cleaning of double-spaces from removals
    text = re.sub(r'  +', ' ', text)
    return text.strip()


## 6. Inference Execution

Running the full pipeline: sampling with activation steering 
followed by deep lexical intervention.


In [0]:
class TestDataset(Dataset):
    def __init__(self, path: str):
        self.df = pd.read_csv(path)
    def __len__(self): return len(self.df)
    def __getitem__(self, i): return self.df.iloc[i].prompt

test_file = os.path.join(Config.data_path, Config.test_dataset_name)
loader = DataLoader(TestDataset(test_file), batch_size=Config.batch_size, shuffle=False)

submission = {"prompt": [], "generation": []}
for batch in tqdm(loader, desc="Evasion Pipeline"):
    # generate with high entropy distribution sampling
    gens = model.generate(
        batch, 
        max_new_tokens=128, 
        do_sample=True, 
        temperature=Config.temperature, 
        top_p=Config.top_p,
        top_k=Config.top_k
    )

    # apply lexical interventions to break statistical signatures
    gens = [apply_lexical_intervention(g) for g in gens]

    submission["prompt"].extend(batch)
    submission["generation"].extend(gens)

pd.DataFrame(submission).to_csv("submission.csv", index=False)
print("Final submission.csv saved.")


## 7. Summary and References

This configuration maximizes evasion durch a combined latent and 
lexical defense strategy. By suppressing multiple AI latent 
directions and increasing output entropy, we significantly reduce 
the predictability index that detectors prioritize.

---
**Citation:**
Cyprus AI Training. [Cyprus Spring 2026] Evading AI-Generated Text Detection.
https://kaggle.com/competitions/cyprus-spring-2026-evading-ai-generated-text-detection, 2026. Kaggle.
